# QuantLab Quickstart

研究/调参/自定义策略入口。需先 `pip install -e '.[us]'` 并能联网取数。
见 doc/需求文档.md · doc/架构设计.md · doc/详细设计.md。

In [ ]:
from quantlab.bootstrap import build_app
from quantlab.indicators.technical import add_indicators
from quantlab.stats.conditions import drop_gt, n_down_days
from quantlab.stats.probability import probability
from quantlab.backtest.engine import Backtester
from quantlab.backtest.strategies import DualMAStrategy

dm = build_app()  # 读 config.yaml

In [ ]:
# 取数（本地优先，缺口自动联网补齐）
df = add_indicators(dm.history('US:AAPL', start='2018-01-01'))
df.tail()

In [ ]:
# 条件概率：跌>1.5% 且连跌2天后，次日上涨概率（含基准率/edge）
r = probability(df, drop_gt(0.015) & n_down_days(2), forward=1)
print(f'N={r.n} 上涨概率={r.p_up:.1%} 基准={r.base_rate:.1%} edge={r.edge:+.1%} '
      f'CI=[{r.ci_low:.1%},{r.ci_high:.1%}] 可靠={r.reliable}')

In [ ]:
# 回测
res = Backtester(cost_bps=5).run(df, DualMAStrategy(5, 20))
print(f'总收益={res.total_return:.1%} 夏普={res.sharpe:.2f} 最大回撤={res.max_drawdown:.1%}')
res.equity_curve.plot(title='策略净值');